In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)

/root/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3

DenseIndex.embeddings:  (2107648, 1024)


In [4]:
test_df = pd.read_csv("../data/test_rewrite_001.csv")
test_id_2_query_d = {}
for query_id, query in zip(test_df['query_id'], test_df['query']):
    test_id_2_query_d[query_id] = query

In [5]:
import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

test_multi_question_df = pd.read_csv("../data/test_rewrite_split_question_001.csv")

recall_citations_l = []

query_id_2_query_list = defaultdict(list)

for query_id, query in zip(test_multi_question_df['query_id'], test_multi_question_df['query']):
    query_id_2_query_list[query_id].append(query)

for query_id, query in test_id_2_query_d.items():
    query_id_2_query_list[query_id].append(query) # 完整的问题在最后一个

recall_hits_l = []
for query_id, query_list in query_id_2_query_list.items():
    all_hits = []
    for query in query_list:
        hits1 = dense_index.search_with_score(query, top_k=500)
        hits2 = sparse_index.search_with_score(query, top_k=500)
        hits_merge = hits_utils.merge_hits_with_score_l_by_max(hits1, hits2)
        all_hits = hits_utils.merge_hits_with_score_l_by_max(all_hits, hits_merge)
        
    print(f"[{query_id}] hits_merge.len:", len(all_hits))

    recall_hits_l.append([hit for hit, score in all_hits])
    

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[test_001] hits_merge.len: 1939
[test_002] hits_merge.len: 1731
[test_003] hits_merge.len: 1773
[test_004] hits_merge.len: 1678
[test_005] hits_merge.len: 1594
[test_006] hits_merge.len: 2260
[test_007] hits_merge.len: 2569
[test_008] hits_merge.len: 1452
[test_009] hits_merge.len: 1751
[test_010] hits_merge.len: 1805
[test_011] hits_merge.len: 1820
[test_012] hits_merge.len: 2606
[test_013] hits_merge.len: 1517
[test_014] hits_merge.len: 1559
[test_015] hits_merge.len: 2023
[test_016] hits_merge.len: 1979
[test_017] hits_merge.len: 1623
[test_018] hits_merge.len: 2576
[test_019] hits_merge.len: 1875
[test_020] hits_merge.len: 2096
[test_021] hits_merge.len: 2228
[test_022] hits_merge.len: 1519
[test_023] hits_merge.len: 1362
[test_024] hits_merge.len: 2228
[test_025] hits_merge.len: 1970
[test_026] hits_merge.len: 2093
[test_027] hits_merge.len: 1440
[test_028] hits_merge.len: 1661
[test_029] hits_merge.len: 2529
[test_030] hits_merge.len: 1659
[test_031] hits_merge.len: 1183
[test_03

In [6]:
query_id_l = []
result_citation_l = []

for idx, (query_id, query_list) in tqdm(enumerate(query_id_2_query_list.items()), total=len(query_id_2_query_list)):
    print("===>",query_id)
    ranked_citation_l_l = []

    raw_query = query_list[-1]
    normalized_query_list = query_list[:-1] # 最后一个包含多个问题
    question_l = []
    for q in normalized_query_list:
        _, question = q.rsplit('\n\n', 1)
        question_l.append(question)

    # 先算context和recall_hits的分数
    # context, _ = query_list[0].rsplit('\n\n', 1)
    # hit_with_score_l_context = reranker_utils.rerank_by_batch_chunked2(reranker, context, recall_hits_l[idx])
    # assert(len(hit_with_score_l_context) == len(recall_hits_l[idx]))

    hit_with_score_l_raw = reranker_utils.rerank_by_batch_chunked2(reranker, raw_query, recall_hits_l[idx])
    hit_with_score_l_question = reranker_utils.rerank_by_batch_chunked2(reranker, '\n\n'.join(question_l), recall_hits_l[idx])
    
    
    hit_with_score_l = [(raw_score[0], raw_score[1]+0.5*question_score[1]) for raw_score, question_score in zip(hit_with_score_l_raw, hit_with_score_l_question)]


    # citation_score_l = citation_utils.compute_citation_score_with_court_consideration_sector_pos(hit_with_score_l)
    citation_score_l = citation_utils.compute_citation_score_with_sentence_pos(hit_with_score_l, "log")

    # 对综合分数排序
    sorted_citation_score_l = sorted([(c,score) for c,score in citation_score_l], key=lambda x: x[1], reverse=True)
    
    # 将citation合并
    query_result = [citation for citation, _ in sorted_citation_score_l]
    
    query_result2 = [citation for citation in query_result if citation in court_consideration_d or citation in law_d]
    
    print("query_result2:", len(query_result2))
    
    result_citation_l.append(';'.join(query_result2[:100]))
    query_id_l.append(query_id)
    
    print(f"{query_id} query_result2.len:", len(query_result2))


result_df = pd.DataFrame({'query_id':query_id_l, 'predicted_citations':result_citation_l})
result_df.to_csv("../output/submission_100.csv", index=False)

  0%|          | 0/40 [00:00<?, ?it/s]

===> test_001


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


query_result2: 661
test_001 query_result2.len: 661
===> test_002
query_result2: 684
test_002 query_result2.len: 684
===> test_003
query_result2: 738
test_003 query_result2.len: 738
===> test_004
query_result2: 512
test_004 query_result2.len: 512
===> test_005
query_result2: 1097
test_005 query_result2.len: 1097
===> test_006
query_result2: 1080
test_006 query_result2.len: 1080
===> test_007
query_result2: 787
test_007 query_result2.len: 787
===> test_008
query_result2: 395
test_008 query_result2.len: 395
===> test_009
query_result2: 528
test_009 query_result2.len: 528
===> test_010
query_result2: 378
test_010 query_result2.len: 378
===> test_011
query_result2: 930
test_011 query_result2.len: 930
===> test_012
query_result2: 1092
test_012 query_result2.len: 1092
===> test_013
query_result2: 848
test_013 query_result2.len: 848
===> test_014
query_result2: 228
test_014 query_result2.len: 228
===> test_015
query_result2: 788
test_015 query_result2.len: 788
===> test_016
query_result2: 575


In [13]:
limit=30
sub_df = pd.read_csv("../output/submission_100.csv")
pred_l = []
for query_id, pred in zip(sub_df['query_id'].tolist(), sub_df['predicted_citations'].tolist()):
    pred2 = pred.split(";")[:limit]
    pred_l.append(';'.join(pred2))
new_sub_df = pd.DataFrame({'query_id':sub_df['query_id'].tolist(), 'predicted_citations':pred_l})
new_sub_df.to_csv("../output/submission.csv", index=False)